# Building Python Applications with LLM APIs: Example Solution

**Problem:** an example solution for the Week 9 lab (*Lab.pdf* in *3 Tutorial*). We send prompts to a large language model (LLM) through an API, handle errors and rate limits, and build the three project options from the lab: a **chatbot**, a **text summariser** and a **code generator**.

**How to run:** this notebook runs in your browser on the course website. You need your own free API key from [OpenRouter](https://openrouter.ai/keys). Paste it into the cell in section 2, then use **Run → Run All Cells**. Without a key the notebook still runs: instead of sending each request, it shows the request it would send.

**Expected result:** with a key, every section prints the model's reply. The wording changes from run to run, and the model behind `openrouter/free` can change too. Without a key, every section prints the request (a JSON message list). The saved outputs below show this no-key version.

> **Keep your key private.** Don't share a notebook that contains your key, and delete the key from the cell before you save or download the notebook.

## 1. Settings
OpenRouter accepts the same request format as OpenAI's *Chat Completions* API, so the same code works with many providers. The model name `openrouter/free` picks one of the free models that are available at the moment. `IN_BROWSER` tells us whether the notebook runs on the course website (Pyodide) or in Python on a desktop computer.

In [1]:
import asyncio
import json
import sys

API_URL = "https://openrouter.ai/api/v1/chat/completions"  # OpenAI-compatible endpoint
MODEL = "openrouter/free"  # a free model that is available right now
IN_BROWSER = sys.platform == "emscripten"  # True on the course website
print("Running in the browser:", IN_BROWSER)

Running in the browser: True


## 2. Your API key
On a desktop computer you would keep the key out of your code: store it in a `.env` file and load it with `python-dotenv` (lab step 3):

```python
from dotenv import load_dotenv
import os
load_dotenv()  # reads OPENROUTER_API_KEY=... from the .env file
API_KEY = os.getenv("OPENROUTER_API_KEY", "")
```

The browser has no `.env` file, so here you paste the key into the cell below.

In [2]:
# paste your key between the quotes (it starts with "sk-or-"); leave it empty to see the requests only
API_KEY = ""

## 3. Sending a request, with error handling
`ask_llm` sends a list of chat messages and returns the model's reply as text.
- In the browser it uses `pyfetch` (the browser's own way of sending web requests). On a desktop it uses `requests`, the package from the lab sheet.
- The key goes in the `Authorization` header.
- Error handling (lab step 5): a rate-limit error (status 429) waits and tries again, waiting longer each time. Any other error prints a message instead of stopping the notebook.

The functions are `async` because the browser sends requests in the background. That's why the later cells call them with `await`, which works in Jupyter and on the course website.

In [3]:
async def post_json(url, headers, payload):
    """Send a POST request and return (status code, JSON reply)."""
    if IN_BROWSER:
        from pyodide.http import pyfetch  # the browser's fetch
        response = await pyfetch(url, method="POST", headers=headers, body=json.dumps(payload))
        return response.status, await response.json()
    import requests  # desktop: pip install requests
    response = requests.post(url, headers=headers, json=payload, timeout=60)
    return response.status_code, response.json()


async def ask_llm(messages, max_retries=3):
    """Send a list of chat messages to the model and return its reply as text."""
    payload = {"model": MODEL, "messages": messages}
    if not API_KEY:
        # no key: show the request instead of sending it
        print("No API key, so the request is not sent. It would be:")
        print(json.dumps(payload, indent=2, ensure_ascii=False))
        return "(no reply: add your API key in section 2)"

    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    for attempt in range(1, max_retries + 1):
        try:
            status, reply = await post_json(API_URL, headers, payload)
        except Exception as error:  # e.g. no internet connection
            return f"(request failed: {error})"
        if status == 200:
            return reply["choices"][0]["message"]["content"]
        if status == 429 and attempt < max_retries:
            wait = 2 ** attempt  # rate limit: wait 2 s, then 4 s, ...
            print(f"Rate limit reached, waiting {wait} s (attempt {attempt} of {max_retries})")
            await asyncio.sleep(wait)
            continue
        message = reply.get("error", {}).get("message", reply) if isinstance(reply, dict) else reply
        return f"(error {status}: {message})"

## 4. Testing the connection
A single question checks that the key and the connection work (lab step 3).

In [4]:
answer = await ask_llm([{"role": "user", "content": "In one sentence, what is a large language model?"}])
print(answer)

No API key, so the request is not sent. It would be:
{
  "model": "openrouter/free",
  "messages": [
    {
      "role": "user",
      "content": "In one sentence, what is a large language model?"
    }
  ]
}
(no reply: add your API key in section 2)


## 5. Project option 1: chatbot
The model remembers nothing between requests. A chatbot therefore keeps the whole conversation in a list (`history`) and sends all of it every time. The first message is a **system** message that gives the model its role, as in the *Role-Task-Format* slide of the lecture.

In [5]:
# the system message sets the role; the list grows by two messages per turn
history = [{"role": "system",
            "content": "You are a friendly teaching assistant for a university course on AI and "
                       "data science. Answer in at most three sentences."}]

async def chat(user_input):
    history.append({"role": "user", "content": user_input})
    answer = await ask_llm(history)
    history.append({"role": "assistant", "content": answer})  # remember the reply for the next turn
    return answer

print(await chat("What is the difference between supervised and unsupervised learning?"))

No API key, so the request is not sent. It would be:
{
  "model": "openrouter/free",
  "messages": [
    {
      "role": "system",
      "content": "You are a friendly teaching assistant for a university course on AI and data science. Answer in at most three sentences."
    },
    {
      "role": "user",
      "content": "What is the difference between supervised and unsupervised learning?"
    }
  ]
}
(no reply: add your API key in section 2)


The follow-up question below only makes sense because the earlier messages are sent again with it. Look at the request: it now holds the whole conversation so far.

In [6]:
print(await chat("Give me one example of the second one."))
print("Messages in the history:", len(history))

No API key, so the request is not sent. It would be:
{
  "model": "openrouter/free",
  "messages": [
    {
      "role": "system",
      "content": "You are a friendly teaching assistant for a university course on AI and data science. Answer in at most three sentences."
    },
    {
      "role": "user",
      "content": "What is the difference between supervised and unsupervised learning?"
    },
    {
      "role": "assistant",
      "content": "(no reply: add your API key in section 2)"
    },
    {
      "role": "user",
      "content": "Give me one example of the second one."
    }
  ]
}
(no reply: add your API key in section 2)
Messages in the history: 5


## 6. Project option 2: text summariser
A prompt **template** fills the text into fixed instructions. Following the *Specificity and Clarity* slide, it states the length, the audience and the style.

In [7]:
article = (
    "Reinforcement learning is a branch of machine learning in which an agent learns by trial and error. "
    "The agent observes the state of its environment, takes an action and receives a reward. "
    "Over many episodes it learns which actions lead to the highest total reward. "
    "Q-learning stores this knowledge in a table of state-action values, while deep reinforcement learning "
    "replaces the table with a neural network so that it can handle very large numbers of states."
)

async def summarise(text, sentences=2):
    prompt = (f"Summarise the following text in {sentences} sentences for a first-year student. "
              f"Use plain language.\n\nText:\n{text}")
    return await ask_llm([{"role": "user", "content": prompt}])

print(await summarise(article))

No API key, so the request is not sent. It would be:
{
  "model": "openrouter/free",
  "messages": [
    {
      "role": "user",
      "content": "Summarise the following text in 2 sentences for a first-year student. Use plain language.\n\nText:\nReinforcement learning is a branch of machine learning in which an agent learns by trial and error. The agent observes the state of its environment, takes an action and receives a reward. Over many episodes it learns which actions lead to the highest total reward. Q-learning stores this knowledge in a table of state-action values, while deep reinforcement learning replaces the table with a neural network so that it can handle very large numbers of states."
    }
  ]
}
(no reply: add your API key in section 2)


## 7. Project option 3: code generator
Here the system message gives the role, and the user message gives the task and the format we want back (*Role-Task-Format*).

In [8]:
async def generate_code(task):
    messages = [
        {"role": "system", "content": "You are an experienced Python developer."},
        {"role": "user", "content": f"Task: {task}\nFormat: reply with only the Python code, "
                                    "in one code block, with short comments."},
    ]
    return await ask_llm(messages)

print(await generate_code("write a function that returns the n-th Fibonacci number"))

No API key, so the request is not sent. It would be:
{
  "model": "openrouter/free",
  "messages": [
    {
      "role": "system",
      "content": "You are an experienced Python developer."
    },
    {
      "role": "user",
      "content": "Task: write a function that returns the n-th Fibonacci number\nFormat: reply with only the Python code, in one code block, with short comments."
    }
  ]
}
(no reply: add your API key in section 2)


## 8. Testing and improving prompts
Lab step 5 asks you to test with different inputs and improve the prompts. This cell sends the vague and the refined prompt from the lecture, so you can compare the replies.

In [9]:
vague = "Write a product description for a waterproof watch."
refined = ("Write a two-sentence product description for a children's waterproof watch, "
           "highlighting its durability and fun design.")

for prompt in [vague, refined]:
    print("PROMPT:", prompt)
    print("REPLY:", await ask_llm([{"role": "user", "content": prompt}]))
    print()

PROMPT: Write a product description for a waterproof watch.
No API key, so the request is not sent. It would be:
{
  "model": "openrouter/free",
  "messages": [
    {
      "role": "user",
      "content": "Write a product description for a waterproof watch."
    }
  ]
}
REPLY: (no reply: add your API key in section 2)

PROMPT: Write a two-sentence product description for a children's waterproof watch, highlighting its durability and fun design.
No API key, so the request is not sent. It would be:
{
  "model": "openrouter/free",
  "messages": [
    {
      "role": "user",
      "content": "Write a two-sentence product description for a children's waterproof watch, highlighting its durability and fun design."
    }
  ]
}
REPLY: (no reply: add your API key in section 2)



### Try it
What happens if you send the follow-up question from section 5 on its own, without the history? Try `await ask_llm([{"role": "user", "content": "Give me one example of the second one."}])`.

<details>
<summary>Answer</summary>

Each request is independent, so the model has no idea what "the second one" refers to. It will usually ask what you mean, or guess. The chatbot in section 5 works only because it sends the earlier messages again with every new question. The price is that requests get longer, and so slower and more expensive, as the conversation grows.
</details>